In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

bronze_path     = "abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_directors"
silver_path     = "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_directors"
quarantine_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_quarantine/netflix_directors"
checkpoint_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_checkpoints/netflix_directors"

def process_directors_microbatch(micro_df, batch_id):
    if micro_df.isEmpty():
        return

    if "_rescued_data" not in micro_df.columns:
        micro_df = micro_df.withColumn("_rescued_data", F.lit(None).cast("string"))

    # Read valid parent show_ids from Silver netflix_titles for foreign-key check
    silver_titles_ids = spark.read.format("delta").load(
        "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
    ).select("show_id").distinct()

    # 1. Standardise strings
    staged = (
        micro_df
        .withColumn("show_id", F.trim(F.col("show_id").cast("string")))
        .withColumn(
            "director",
            F.when(F.col("director").isNull() | (F.trim(F.col("director")) == ""), F.lit(None).cast("string"))
             .otherwise(F.trim(F.regexp_replace(F.col("director"), r"\s+", " ")))
        )
    )

    # 2. Check foreign-key match against Silver netflix_titles
    staged_with_fk = (
        staged.alias("d")
        .join(silver_titles_ids.alias("t"), F.col("d.show_id") == F.col("t.show_id"), "left")
        .select(
            F.col("d.*"),
            F.when(F.col("t.show_id").isNull(), F.lit(True)).otherwise(F.lit(False)).alias("_is_orphan")
        )
    )

    # 3. Validation rules
    invalid_condition = (
        F.col("_rescued_data").isNotNull() |
        F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$") |
        F.col("director").isNull() | F.col("director").rlike(r"^[0-9\W_]+$") |
        F.col("_is_orphan")
    )

    quarantine_df = (
        staged_with_fk.filter(invalid_condition)
        .withColumn(
            "quarantine_reason",
            F.when(F.col("_rescued_data").isNotNull(), "Corrupt CSV / Rescued Data")
             .when(F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$"), "Invalid or Missing show_id")
             .when(F.col("director").isNull() | F.col("director").rlike(r"^[0-9\W_]+$"), "Invalid or Missing director")
             .when(F.col("_is_orphan"), "Orphan show_id (missing in silver.netflix_titles)")
             .otherwise("Unknown validation failure")
        )
        .drop("_is_orphan")
        .withColumn("_quarantined_at", F.current_timestamp())
    )

    clean_df = (
        staged_with_fk.filter(~invalid_condition)
        .drop("_is_orphan", "_rescued_data")
        .dropDuplicates(["show_id", "director"])
        .withColumn("_silver_processed_at", F.current_timestamp())
    )

    # 4. Write only new batch to Silver (Upsert) and Quarantine (Append)
    if not clean_df.isEmpty():
        if DeltaTable.isDeltaTable(spark, silver_path):
            (
                DeltaTable.forPath(spark, silver_path).alias("target")
                .merge(
                    clean_df.alias("source"),
                    "target.show_id = source.show_id AND target.director = source.director"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
        else:
            clean_df.write.format("delta").mode("overwrite").save(silver_path)

    if not quarantine_df.isEmpty():
        quarantine_df.write.format("delta").mode("append").option("mergeSchema", "true").save(quarantine_path)

# Stream ONLY new Bronze commits using Structured Streaming
query = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
    .writeStream
    .foreachBatch(process_directors_microbatch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print("✅ Incremental Silver processing complete for netflix_directors.")
